# OMNI-X16 / OMNI-902 · CCD's clip vectors for the forecasting model, with both encoders (Colab GPU)

Runs the repo's own code from a clone of the branch: `scripts/encode_pfm_windows.py` (the grid and the runner),
ai_workers' InternVideo2 and window functions, and `scripts/encode_virat_cosmos.py`'s Cosmos encoder.

**What it makes:** for every window of `datasets/windows/ccd.csv` (4,500 videos, every split), a clip vector every
0.5 s from the window's last frame back to the video's start: **39,233 clips per encoder**. Each clip is Algorithm 2's
(8 frames from the 1 s before `t`). One file per video on Drive, so a lost session resumes where it stopped.

**Two processes, two transformers versions (D-50):** InternVideo2 runs here on the repo's 5.20.1; Cosmos-Embed1 runs
in a process of its own with transformers 4.51.3 first on its path.

1. **Setup:** clone, install the pinned versions; transformers 4.51.3 into `/content/cosmos_env` for Cosmos only
2. **CCD's videos** copied from Drive to this machine once (both encoders read them)
3. **InternVideo2** → `MyDrive/omnisight/embeddings/pfm/ccd/internvideo2/`
4. **Cosmos-Embed1** (its own process) → `MyDrive/omnisight/embeddings/pfm/ccd/cosmos/`
5. **Check and save:** every video done in both, the time per clip → `MyDrive/omnisight/results/x16_ccd_encoding.json`

Runtime → Change runtime type → **GPU** (A100 or L4). Then Run all. If the session ends, Run all again: finished
videos are skipped.

In [ ]:
# 1 · Setup
BRANCH = "slice-09-10-forecast"
REPO = "https://github.com/OmniSight-team/omnisight.git"

import os, subprocess, sys
if not os.path.isdir("/content/omni-sight"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, "/content/omni-sight"], check=True)
else:
    subprocess.run(["git", "-C", "/content/omni-sight", "pull", "-q"], check=True)
COMMIT = subprocess.run(["git", "-C", "/content/omni-sight", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print(COMMIT)

# This process: the versions in uv.lock; Colab keeps its own GPU build of torch and torchvision.
!pip install -q "transformers==5.20.1" "huggingface-hub==2.2.0" "safetensors==0.8.0" "open-clip-torch==3.3.0" -e /content/omni-sight/services/common
# The Cosmos process only: transformers 4.51.3 and what it needs that Colab lacks, in a folder of its own.
!pip install -q --no-deps --target /content/cosmos_env "transformers==4.51.3" "tokenizers>=0.21,<0.22" "huggingface-hub>=0.30,<1.0" einops

sys.path.insert(0, "/content/omni-sight/services/common")
sys.path.insert(0, "/content/omni-sight/services/ai_workers")
sys.path.insert(0, "/content/omni-sight/scripts")
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import torch
assert torch.cuda.is_available(), "no GPU: Runtime -> Change runtime type -> GPU"
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# 2 · CCD's videos, copied from Drive to this machine once (many small files: in parallel)
import shutil, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import encode_pfm_windows as pfm

REPO_DIR = Path("/content/omni-sight")
DRIVE = Path("/content/drive/MyDrive/omnisight")
CCD_DRIVE = DRIVE / "data" / "ccd"
CCD = Path("/content/ccd")
WINDOWS = pfm.read_windows(REPO_DIR / "datasets" / "windows" / "ccd.csv")
assert len(WINDOWS) == 4500

def copy(rel):
    target = CCD / rel
    if not target.is_file():
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(CCD_DRIVE / rel, target)

start = time.time()
rels = [rows[0]["video_relpath"] for rows in WINDOWS.values()]
with ThreadPoolExecutor(16) as pool:
    list(pool.map(copy, rels))
missing = [r for r in rels if not (CCD / r).is_file()]
print(f"{len(rels) - len(missing)} of {len(rels)} videos here, {time.time() - start:.0f} s")
assert not missing, missing[:5]
CLIPS = sum(len(pfm.grid_frames(rows, 10.0, pfm.STEPS_S["ccd"])) for rows in WINDOWS.values())
print(f"{CLIPS:,} grid clips per encoder")

In [ ]:
# 3 · InternVideo2 (the product's pin, D-33), float32, in this process
import json
from omnisight_common.settings import CommonSettings
from app.encoder import ensure_checkpoint
from app import clip_encoder
from app.clip_encoder import ClipTextEncoder

MODELS = DRIVE / "models" / "pretrained"
OUT = DRIVE / "embeddings" / "pfm" / "ccd"
SESSION = time.strftime("%Y-%m-%d %H:%M") + " " + torch.cuda.get_device_name(0)
iv2_pins = {name: CommonSettings.model_fields[name].default for name in ("clip_model_id", "clip_model_revision")}
iv2_dir = ensure_checkpoint(iv2_pins["clip_model_id"], iv2_pins["clip_model_revision"], MODELS / "internvideo2-clip-s", clip_encoder.ALLOW_PATTERNS)
iv2 = ClipTextEncoder(iv2_dir, device="cuda")
manifest = pfm.run(WINDOWS, CCD, OUT / "internvideo2", iv2, clip_encoder, iv2_pins, pfm.STEPS_S["ccd"], session=SESSION)
print("failed:", manifest["failed"])
del iv2
torch.cuda.empty_cache()

In [ ]:
# 4 · Cosmos-Embed1-336p (X14's pin), BF16, in its own process on transformers 4.51.3 (D-50)
import encode_virat_cosmos as cosmos_runner

cosmos_dir = cosmos_runner.download(MODELS / "cosmos-embed1-336p")  # on Drive since X14
cosmos_pins = {"clip_model_id": cosmos_runner.COSMOS_ID, "clip_model_revision": cosmos_runner.COSMOS_REVISION}
proc = subprocess.Popen(
    [sys.executable, str(REPO_DIR / "scripts" / "encode_pfm_windows.py"), "cosmos",
     "--repo", str(REPO_DIR), "--model-dir", str(cosmos_dir), "--dataset", "ccd", "--data", str(CCD),
     "--out", str(OUT / "cosmos"), "--pins", json.dumps(cosmos_pins), "--session", SESSION],
    env={**os.environ, "PYTHONPATH": "/content/cosmos_env"},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
assert proc.wait() == 0, "the Cosmos process failed: see its output above"

In [ ]:
# 5 · Check both folders and save the summary
summaries = {name: pfm.summary(WINDOWS, OUT / name) for name in ("internvideo2", "cosmos")}
for name, s in summaries.items():
    print(f"{name:13s} {s['done']} of {s['videos']} videos, {s['clips']:,} clips, {s['ms_per_clip']} ms per clip "
          f"in the encoder, {s['encoder_seconds'] / 3600:.2f} h; failed {len(s['failed'])}: {list(s['failed'].items())[:3]}")
    assert s["done"] == 4500 and s["clips"] == CLIPS and not s["failed"]

out = DRIVE / "results" / "x16_ccd_encoding.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps({"commit": COMMIT, "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
                           "clips_expected": CLIPS, "encoders": summaries}, indent=1))
print(out)